In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
# Import packages and modules
import numpy as np
import torch
from RL4CRN_Feedback.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN_Feedback.Policies.BimolecularMassActionPolicy import BimolecularMassActionPolicy
from RL4CRN_Feedback.Utils.Utils import batch_multi_hot

In [3]:
# Construct the basic CRN
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3', 'u_4']
stoichiometry_reactants = np.array([[0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [1], [0]], dtype=np.int8)
k = 1
parameters = np.array([k], dtype=np.float32)
input_influence_matrix = np.array([[0], [0], [0], [0]], dtype=np.int8)
outputs = np.array([1], dtype=np.int8)
IOCRN_0 = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)

# Add reactions to CRN_1
IOCRN_1 = IOCRN_0.clone()
IOCRN_1.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 0, 'product2 index': 0, 'input influence index': 0, 'rate constant':0.1}, mode='species index')
IOCRN_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.5}, mode='species index')
IOCRN_1.add_reaction({'reactant1 index': 1, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 2, 'rate constant':0.7}, mode='species index')

# Add reactions to CRN_2
IOCRN_2 = IOCRN_0.clone()
IOCRN_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 1, 'input influence index': 2, 'rate constant':0.1}, mode='species index')
IOCRN_2.add_reaction({'reactant1 index': 2, 'reactant2 index': 3, 'product1 index': 0, 'product2 index': 3, 'input influence index': 4, 'rate constant':0.1}, mode='species index')
IOCRN_2.add_reaction({'reactant1 index': 0, 'reactant2 index': 1, 'product1 index': 2, 'product2 index': 2, 'input influence index': 0, 'rate constant':0.3}, mode='species index')

# Add reactions to CRN_3
IOCRN_3 = IOCRN_0.clone()
IOCRN_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 0, 'product1 index': 0, 'product2 index': 3, 'input influence index': 0, 'rate constant':0.8}, mode='species index')
IOCRN_3.add_reaction({'reactant1 index': 3, 'reactant2 index': 3, 'product1 index': 1, 'product2 index': 3, 'input influence index': 3, 'rate constant':0.2}, mode='species index')
IOCRN_3.add_reaction({'reactant1 index': 0, 'reactant2 index': 2, 'product1 index': 1, 'product2 index': 1, 'input influence index': 3, 'rate constant':0.9}, mode='species index')

# Create list of CRNs to represent a batch
IOCRN_list = [IOCRN_1, IOCRN_2, IOCRN_3]

In [4]:
# Construct the Policy Model
num_species = 3; num_inputs = 4
encoder_attributes = {"hidden_size": 64, "num_layers": 2}
structure_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
rate_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
input_influence_decoder_attributes = {"hidden_size": 64, "num_layers": 2}
hidden_size = 1024
num_possible_reactions = IOCRN_0.get_reactions_range()
PolicyModel = BimolecularMassActionPolicy(num_possible_reactions, num_inputs, encoder_attributes, hidden_size, structure_decoder_attributes, rate_decoder_attributes, input_influence_decoder_attributes, continuous_distribution='lognormal', allow_input_influence=True)

In [5]:
# Collect observations from the batch of CRNs
batch_size = len(IOCRN_list)
reactions_indices_batch = np.array([CRN.reactions_indices for CRN in IOCRN_list])
parameters_batch = np.array([CRN.parameters for CRN in IOCRN_list])

def get_influenced_reactions_batch(IOCRN_list, num_inputs):
    influenced_reactions = []
    for i in range(num_inputs):
        rows = [np.array(crn.list_influenced_reactions[i]) for crn in IOCRN_list]
        max_len = max((len(r) for r in rows), default=0)
        padded = [np.pad(r, (0, max_len - len(r)), constant_values=0) for r in rows]
        influenced_reactions.append(np.array(padded).astype(np.uint64))
    return influenced_reactions

reactions_indices_influenced_by_inputs_batch = get_influenced_reactions_batch(IOCRN_list, num_inputs)

observation_batch = reactions_indices_batch, parameters_batch, reactions_indices_influenced_by_inputs_batch

In [6]:
# Run the Policy Model
samples, log_probability, entropy = PolicyModel(observation_batch, mode='full')

# Print the output
for i in range(batch_size):
    print(f"Sample {i}:")
    print("Generated reaction structure:", samples[i]['reaction index'])
    print("Generated reaction rates:", samples[i]['rate constant'])
    print("Generated input influence:", samples[i]['input influence index'])
    print("log probability:", log_probability)
    print("Entropy:", entropy)
    print("---------------------")

Sample 0:
Generated reaction structure: 82
Generated reaction rates: 3.9876657
Generated input influence: 3
log probability: tensor([-9.2760, -6.4835, -6.6995], grad_fn=<AddBackward0>)
Entropy: tensor([7.4198, 7.4194, 7.4221], grad_fn=<AddBackward0>)
---------------------
Sample 1:
Generated reaction structure: 20
Generated reaction rates: 0.4186737
Generated input influence: 0
log probability: tensor([-9.2760, -6.4835, -6.6995], grad_fn=<AddBackward0>)
Entropy: tensor([7.4198, 7.4194, 7.4221], grad_fn=<AddBackward0>)
---------------------
Sample 2:
Generated reaction structure: 70
Generated reaction rates: 0.39020607
Generated input influence: 1
log probability: tensor([-9.2760, -6.4835, -6.6995], grad_fn=<AddBackward0>)
Entropy: tensor([7.4198, 7.4194, 7.4221], grad_fn=<AddBackward0>)
---------------------


In [7]:
# Add samples to the CRNs
for i in range(batch_size):
    # reaction is a dictionary with keys 'reaction index', 'input influence index', 'rate constant'
    reaction = samples[i]
    print('Before adding the generated reactions:')
    IOCRN_list[i].print_reactions()
    IOCRN_list[i].add_reaction(reaction, mode='reaction index')
    print('After adding the generated reactions:')
    IOCRN_list[i].print_reactions()
    print('----------------------')
    print('---------------------------------------------------')

Before adding the generated reactions:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> Z_1 + Z_2 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + Z_2 -> Z_2 ; Rate Constant: 0.7u_2 

After adding the generated reactions:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> 0 ; Rate Constant: 0.1 
Reaction 2: 2 X_1 -> Z_1 + Z_2 ; Rate Constant: 0.5u_3 
Reaction 3: X_1 + Z_2 -> Z_2 ; Rate Constant: 0.7u_2 
Reaction 4: 2 Z_2 -> 0 ; Rate Constant: 3.9876656532287598u_3 

----------------------
---------------------------------------------------
Before adding the generated reactions:
Inputs: ['u_1', 'u_2', 'u_3', 'u_4'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
